In [14]:
import sys
from pathlib import Path
import numpy as np

ROOT = Path.cwd().parent                 # notebooks/ -> ml-project1/
sys.path.insert(0, str(ROOT))

d = np.load(ROOT / "cache" / "dati_puliti.npz")
x_base, x_base_test = d["x_base"], d["x_base_test"]
cols_base = list(d["cols_base"])
y_train, test_ids = d["y_train"], d["test_ids"]

print("train:", x_base.shape, "  test:", x_base_test.shape)
print("positivi: %d (%.2f%%)" % ((y_train == 1).sum(), 100 * (y_train == 1).mean()))

train: (328135, 95)   test: (109379, 95)
positivi: 28975 (8.83%)


### **creazione fold in maniera uniforme**

In [15]:
def fold_stratificati(y, k=5, seed=1):
    """Divide gli indici in k fold con la stessa proporzione di positivi (y == 1)."""
    rng = np.random.default_rng(seed)
    pos = rng.permutation(np.where(y == 1)[0])          # positivi mescolati
    neg = rng.permutation(np.where(y != 1)[0])          # negativi mescolati
    return [np.sort(np.concatenate([p, n]))
            for p, n in zip(np.array_split(pos, k), np.array_split(neg, k))]

def split_fold(folds, k):
    """Indici di training e di validation per il fold k."""
    val = folds[k]
    train = np.concatenate([folds[j] for j in range(len(folds)) if j != k])
    return train, val

folds = fold_stratificati(y_train, k=5, seed=1)

# --- controlli ---
print(f"{'fold':>4s} {'persone':>8s} {'infarti':>8s} {'% infarti':>10s}")
for k, f in enumerate(folds):
    print(f"{k+1:4d} {len(f):8d} {(y_train[f] == 1).sum():8d} {100*(y_train[f] == 1).mean():9.2f}%")

tutti = np.concatenate(folds)
print("\nogni persona in esattamente un fold:", len(tutti) == len(y_train) and len(np.unique(tutti)) == len(y_train))
tr, va = split_fold(folds, 0)
print("fold 1 -> training:", len(tr), "  validation:", len(va), "  sovrapposizione:", len(np.intersect1d(tr, va)))

fold  persone  infarti  % infarti
   1    65627     5795      8.83%
   2    65627     5795      8.83%
   3    65627     5795      8.83%
   4    65627     5795      8.83%
   5    65627     5795      8.83%

ogni persona in esattamente un fold: True
fold 1 -> training: 262508   validation: 65627   sovrapposizione: 0


## Step 2: metriche di valutazione

Con l'8,8% di infarti, l'**accuracy** da sola è ingannevole: un modello che predice sempre
 
 "nessun infarto" ha il 91,2% di accuracy senza aver imparato nulla. Per questo misuriamo 
 
 anche precision, recall ed F1, partendo dalla **matrice di confusione**:

| | vero infarto (1) | vero sano (0) |
|---|---|---|
| **predetto infarto** | TP (veri positivi) | FP (falsi positivi) |
| **predetto sano** | FN (falsi negativi) | TN (veri negativi) |

$$\text{accuracy} = \frac{TP + TN}{N} \qquad \text{precision} = \frac{TP}{TP + FP}
 \qquad \text{recall} = \frac{TP}{TP + FN} \qquad F_1 = \frac{2\,TP}{2\,TP + FP + FN}$$

- **precision**: dei casi predetti come infarto, quanti lo sono davvero;
- **recall**: dei veri infarti, quanti il modello ne trova;
- **F1**: media armonica delle due. È alta solo se lo sono entrambe.

Se il modello non predice nessun positivo, precision ed F1 valgono 0 (evitiamo la divisione per zero).

La funzione accetta etichette sia in {-1, 1} sia in {0, 1}: considera positivo tutto ciò che vale 1.

### Baseline di riferimento
Per capire se un modello impara qualcosa, lo confrontiamo con tre predittori "stupidi":
| predittore | accuracy | F1 |
|---|---|---|
| sempre "sano" | 91,2% | 0 |
| sempre "infarto" | 8,8% | ~16% |
| a caso, con l'8,8% di "infarto" | ~84% | ~9% |

Un modello utile deve superare nettamente l'F1 di questi.

## Step 2: metriche di valutazione

Ogni persona finisce in una delle 4 caselle della **matrice di confusione**:

| | vero infarto (1) | vero sano |
|---|---|---|
| **predetto infarto** | TP: veri positivi (trovati) | FP: falsi positivi (falsi allarmi) |
| **predetto sano** | FN: falsi negativi (sfuggiti) | TN: veri negativi |

Da questi quattro numeri:

| metrica | formula | domanda a cui risponde |
|---|---|---|
| accuracy | $\frac{TP + TN}{N}$ | quante risposte giuste in tutto? |
| precision | $\frac{TP}{TP + FP}$ | quando il modello dice "infarto", quanto spesso ha ragione? |
| recall | $\frac{TP}{TP + FN}$ | dei veri infarti, quanti ne trova? |
| F1 | $\frac{2\,TP}{2\,TP + FP + FN}$ | equilibrio tra precision e recall (media armonica) |

- Con l'8,8% di infarti l'**accuracy inganna**: "tutti sani" ottiene il 91,2%. L'**F1** non usa i TN, quindi quel trucco ottiene F1 = 0.
- Se il modello non predice nessun positivo, precision ed F1 valgono 0 (evitiamo la divisione per zero).
<!-- - Le funzioni accettano etichette in {-1, 1} o {0, 1}: è positivo tutto ciò che vale 1. -->

In [16]:
def matrice_confusione(y_vero, y_pred):
    """Restituisce (TP, FP, FN, TN). Positivo = valore 1."""
    vero = np.asarray(y_vero) == 1
    pred = np.asarray(y_pred) == 1
    TP = int(np.sum(vero & pred))       # infarto vero, predetto infarto
    FP = int(np.sum(~vero & pred))      # sano, ma predetto infarto (falso allarme)
    FN = int(np.sum(vero & ~pred))      # infarto vero, ma predetto sano (sfuggito)
    TN = int(np.sum(~vero & ~pred))     # sano, predetto sano
    return TP, FP, FN, TN

def accuracy(TP, FP, FN, TN):
    return (TP + TN) / (TP + FP + FN + TN)

def precision(TP, FP, FN, TN):
    return TP / (TP + FP) if TP + FP > 0 else 0.0

def recall(TP, FP, FN, TN):
    return TP / (TP + FN) if TP + FN > 0 else 0.0

def f1(TP, FP, FN, TN):
    return 2 * TP / (2 * TP + FP + FN) if TP > 0 else 0.0

def metriche(y_vero, y_pred):
    """Tutte le metriche in un dizionario."""
    c = matrice_confusione(y_vero, y_pred)
    TP, FP, FN, TN = c
    return {"TP": TP, "FP": FP, "FN": FN, "TN": TN,
            "accuracy": accuracy(*c), "precision": precision(*c),
            "recall": recall(*c), "f1": f1(*c)}

def stampa_metriche(m, titolo=""):
    print(f"{titolo:24s} acc {m['accuracy']:.4f}  prec {m['precision']:.4f}  "
          f"rec {m['recall']:.4f}  F1 {m['f1']:.4f}   "
          f"(TP {m['TP']}, FP {m['FP']}, FN {m['FN']}, TN {m['TN']})")


In [17]:
def scegli_soglia(y, punteggi, metrica="f1", percentili=np.linspace(50, 99.5, 200)):
    """Soglia che massimizza la metrica (calcolata su y e punteggi del TRAINING)."""
    candidate = np.unique(np.percentile(punteggi, percentili))
    valori = [metriche(y, (punteggi >= t).astype(int))[metrica] for t in candidate]
    k = int(np.argmax(valori))
    return candidate[k], valori[k]

def predici(punteggi, soglia):
    """Classificazione 0/1 con la soglia scelta."""
    return (punteggi >= soglia).astype(int)

# --- test con punteggi finti: i positivi hanno in media un punteggio più alto ---
rng = np.random.default_rng(0)
y_finto = (rng.random(20000) < 0.088).astype(int)
punteggi_finti = rng.normal(0, 1, 20000) + 1.5 * y_finto
t, f = scegli_soglia(y_finto, punteggi_finti)
print(f"soglia scelta {t:.3f}   F1 {f:.4f}   persone segnalate {100*(punteggi_finti >= t).mean():.1f}%")
for tt in [0.0, 0.5, 1.0, 1.5, 2.0]:
    m = metriche(y_finto, predici(punteggi_finti, tt))
    print(f"   soglia {tt:.1f}: prec {m['precision']:.3f}  rec {m['recall']:.3f}  F1 {m['f1']:.3f}")

soglia scelta 1.634   F1 0.4605   persone segnalate 8.7%
   soglia 0.0: prec 0.155  rec 0.928  F1 0.265
   soglia 0.5: prec 0.210  rec 0.829  F1 0.335
   soglia 1.0: prec 0.299  rec 0.683  F1 0.416
   soglia 1.5: prec 0.418  rec 0.493  F1 0.452
   soglia 2.0: prec 0.556  rec 0.308  F1 0.396


In [18]:
from preprocessing import fit_transform_prep, transform_prep
from implementations import (least_squares, ridge_regression, mean_squared_error_gd,
                             mean_squared_error_sgd, logistic_regression, reg_logistic_regression,
                             compute_loss, calculate_logistic_loss, sigmoid)

# prova: dalle 95 colonne alla matrice delle covariate, sul fold 1
tr, va = split_fold(folds, 0)
p, X_tr = fit_transform_prep(x_base[tr], cols_base, verbose=True)
X_va = transform_prep(x_base[va], p)
print("da", x_base.shape[1], "colonne a", X_tr.shape[1], "  train", X_tr.shape, " val", X_va.shape,
      "  NaN:", np.isnan(X_tr).any() or np.isnan(X_va).any())

colonne costanti tolte: 0 []
colonne identiche tolte: 3
   NaN[_BMI5]             = _BMI5CAT=NaN  (tenuta)
   NaN[DROCDY3_]          = DRNKANY5=NaN  (tenuta)
   PAMISS1_=NaN           = EXERANY2=NaN  (tenuta)
da 95 colonne a 256   train (262508, 256)  val (65627, 256)   NaN: False


## Step 4: cross-validation dei 6 metodi

Per ogni fold:
1. `fit_transform_prep` sul training → matrice delle covariate (~256 colonne); `transform_prep` sul validation;
2. ogni metodo di `implementations.py` viene addestrato sul training, misurando il **tempo**;
3. punteggi: $s = x^\top w$ (modelli lineari) o $s = \sigma(x^\top w)$ (logistica);
4. la **soglia** si sceglie sul training (massimo F1), poi si misurano le metriche sul validation.

Target in {0, 1} per tutti i metodi: $y^{01} = (y + 1)/2$.

**Passo dei metodi a gradiente.** Per la MSE la discesa del gradiente converge se $\gamma \le 1/L$, dove $L$ è il più grande autovalore di $X^\top X / N$ (lo stimiamo con il metodo delle potenze). Per la logistica la curvatura è al massimo $L/4$, quindi $\gamma = 4/L$.
La SGD usa un solo dato per passo (come richiesto): passo piccolo e fisso, un'epoca.

**Come leggere i risultati:**
- **tempo**: costo computazionale di ogni metodo;
- **loss train** di GD e SGD rispetto a `least_squares` (che è l'ottimo esatto della MSE): se è uguale, il metodo iterativo è arrivato a convergenza;
- **F1 train vs F1 val**: se sono vicini non c'è overfitting;
- **F1 val ± dev. std**: la deviazione standard tra i fold è il rumore. Due metodi sono davvero diversi solo se la differenza è più grande del rumore.

In [19]:
import time
# =====================================================================
# Step 4: CV a 5 fold con i 6 metodi di implementations.py
# =====================================================================
FOLD_DA_USARE = range(len(folds))   # per una prima prova veloce: range(1)
MAX_ITERS    = 200        # passi di GD (MSE e logistica)
SGD_ITERS    = None       # passi di SGD: None = un'epoca (tanti passi quante righe del training)
GAMMA_SGD    = 1e-3       # passo della SGD (un solo dato per passo)
LAMBDA_RIDGE = 1e-6       # regolarizzazione: valori provvisori, da scegliere più avanti in CV
LAMBDA_LOG   = 1e-6

y01 = (y_train == 1).astype(np.float64)          # target in {0, 1} per tutti i metodi

def costante_lipschitz(X, iters=30, seed=0):
    """Autovalore massimo di X^T X / N (metodo delle potenze): il passo di GD sicuro è 1/L."""
    v = np.random.default_rng(seed).standard_normal(X.shape[1])
    for _ in range(iters):
        v = X.T @ (X @ v) / X.shape[0]
        lam = np.linalg.norm(v)
        v /= lam
    return lam

def _sgd(y, X, L):
    np.random.seed(0)                              # SGD riproducibile
    it = len(y) if SGD_ITERS is None else SGD_ITERS
    return mean_squared_error_sgd(y, X, np.zeros(X.shape[1]), it, GAMMA_SGD)

METODI = {
    "least_squares": lambda y, X, L: least_squares(y, X),
    "ridge":         lambda y, X, L: ridge_regression(y, X, LAMBDA_RIDGE),
    "GD (MSE)":      lambda y, X, L: mean_squared_error_gd(y, X, np.zeros(X.shape[1]), MAX_ITERS, 1 / L),
    "SGD (MSE)":     _sgd,
    "logistica":     lambda y, X, L: logistic_regression(y, X, np.zeros(X.shape[1]), MAX_ITERS, 4 / L),
    "logistica reg": lambda y, X, L: reg_logistic_regression(y, X, LAMBDA_LOG, np.zeros(X.shape[1]), MAX_ITERS, 4 / L),
}
LOGISTICI = {"logistica", "logistica reg"}

risultati = {nome: [] for nome in METODI}
for k in FOLD_DA_USARE:
    tr, va = split_fold(folds, k)
    t0 = time.perf_counter()
    p, X_tr = fit_transform_prep(x_base[tr], cols_base)
    X_va = transform_prep(x_base[va], p)
    X_tr, X_va = X_tr.astype(np.float64), X_va.astype(np.float64)   # evita conversioni a ogni passo
    y_tr, y_va = y01[tr], y01[va]
    L = costante_lipschitz(X_tr)
    print(f"fold {k+1}: X_tr {X_tr.shape}, preprocessing {time.perf_counter() - t0:.1f}s, L = {L:.2f}")

    for nome, allena in METODI.items():
        t0 = time.perf_counter()
        w, loss_tr = allena(y_tr, X_tr, L)
        tempo = time.perf_counter() - t0
        s_tr, s_va = X_tr @ w, X_va @ w                       # punteggi
        if nome in LOGISTICI:
            s_tr, s_va = sigmoid(s_tr), sigmoid(s_va)
            loss_va = calculate_logistic_loss(y_va, X_va, w)
        else:
            loss_va = compute_loss(y_va, X_va, w)
        soglia, f1_tr = scegli_soglia(y_tr, s_tr)              # soglia scelta SUL TRAINING
        m = metriche(y_va, (s_va >= soglia).astype(int))       # metriche SUL VALIDATION
        risultati[nome].append(dict(tempo=tempo, loss_tr=loss_tr, loss_va=loss_va,
                                    soglia=soglia, f1_tr=f1_tr, **m))
        print(f"   {nome:14s} {tempo:7.2f}s   F1 val {m['f1']:.4f}")

# --- riepilogo: media (± dev. std) sui fold ---
print(f"\n{'metodo':14s} {'tempo [s]':>9s} {'loss tr':>8s} {'loss val':>8s} {'soglia':>7s} "
      f"{'F1 tr':>6s} {'F1 val':>15s} {'prec':>6s} {'rec':>6s} {'acc':>6s}")
for nome, r in risultati.items():
    media = lambda c: np.mean([x[c] for x in r])
    f1v = np.array([x["f1"] for x in r])
    print(f"{nome:14s} {media('tempo'):9.2f} {media('loss_tr'):8.4f} {media('loss_va'):8.4f} "
          f"{media('soglia'):7.3f} {media('f1_tr'):6.3f} {f1v.mean():8.4f} ± {f1v.std():.4f} "
          f"{media('precision'):6.3f} {media('recall'):6.3f} {media('accuracy'):6.3f}")

fold 1: X_tr (262508, 256), preprocessing 9.6s, L = 8.29
   least_squares     0.45s   F1 val 0.4223
   ridge             0.66s   F1 val 0.4222
   GD (MSE)          9.68s   F1 val 0.4180
   SGD (MSE)         6.62s   F1 val 0.4037
   logistica        11.44s   F1 val 0.4046
   logistica reg    11.24s   F1 val 0.4046
fold 2: X_tr (262508, 256), preprocessing 10.0s, L = 8.28
   least_squares     0.32s   F1 val 0.4217
   ridge             0.47s   F1 val 0.4216
   GD (MSE)          9.47s   F1 val 0.4169
   SGD (MSE)         6.83s   F1 val 0.4079
   logistica        10.96s   F1 val 0.3993
   logistica reg    11.30s   F1 val 0.4014
fold 3: X_tr (262508, 256), preprocessing 9.7s, L = 8.26
   least_squares     0.29s   F1 val 0.4173
   ridge             0.48s   F1 val 0.4171
   GD (MSE)          9.38s   F1 val 0.4135
   SGD (MSE)         6.47s   F1 val 0.4005
   logistica        11.36s   F1 val 0.3961
   logistica reg    11.30s   F1 val 0.3964
fold 4: X_tr (262508, 256), preprocessing 10.2s, L = 8

In [20]:
# =====================================================================
# Convergenza della logistica (fold 1): passi più lunghi e più iterazioni
# =====================================================================
tr, va = split_fold(folds, 0)
p, X_tr = fit_transform_prep(x_base[tr], cols_base)
X_va = transform_prep(x_base[va], p)
X_tr, X_va = X_tr.astype(np.float64), X_va.astype(np.float64)
y_tr, y_va = y01[tr], y01[va]
L = costante_lipschitz(X_tr)

print(f"{'gamma':>8s} {'passi':>6s} {'tempo':>7s} {'loss tr':>9s} {'F1 val':>7s}")
for fattore in [4, 16, 64]:                    # gamma = fattore / L
    gamma = fattore / L
    w, passi, tempo = np.zeros(X_tr.shape[1]), 0, 0.0
    for blocco in [100, 200, 700]:             # passi cumulati: 100, 300, 1000
        t0 = time.perf_counter()
        w, loss = logistic_regression(y_tr, X_tr, w, blocco, gamma)   # riparte dall'ultimo w
        tempo += time.perf_counter() - t0
        passi += blocco
        soglia, _ = scegli_soglia(y_tr, sigmoid(X_tr @ w))
        f1_va = metriche(y_va, (sigmoid(X_va @ w) >= soglia).astype(int))["f1"]
        print(f"{fattore:>5d}/L {passi:6d} {tempo:6.1f}s {loss:9.5f} {f1_va:7.4f}")

   gamma  passi   tempo   loss tr  F1 val
    4/L    100    5.5s   0.23919  0.3862
    4/L    300   16.7s   0.22811  0.4114
    4/L   1000   55.9s   0.22265  0.4244
   16/L    100    5.7s   0.22628  0.4142
   16/L    300   17.4s   0.22224  0.4256
   16/L   1000   57.1s   0.22090  0.4273
   64/L    100    5.6s   0.47900  0.4142
   64/L    300   16.8s   0.98733  0.1157
   64/L   1000   56.0s   0.37005  0.3502


In [22]:
MAX_ITERS = 1000
METODI = {
    "least_squares": lambda y, X, L: least_squares(y, X),
    "logistica":     lambda y, X, L: logistic_regression(y, X, np.zeros(X.shape[1]), MAX_ITERS, 16 / L),
}

In [23]:
import time
# =====================================================================
# CV a 5 fold: least_squares vs logistica (gamma = 16/L, 1000 passi)
# =====================================================================
MAX_ITERS    = 1000
FATTORE_PASSO = 16        # gamma = FATTORE_PASSO / L  (64 diverge, 4 è troppo lento)

y01 = (y_train == 1).astype(np.float64)          # target in {0, 1}

def costante_lipschitz(X, iters=30, seed=0):
    """Autovalore massimo di X^T X / N (metodo delle potenze)."""
    v = np.random.default_rng(seed).standard_normal(X.shape[1])
    for _ in range(iters):
        v = X.T @ (X @ v) / X.shape[0]
        lam = np.linalg.norm(v)
        v /= lam
    return lam

METODI = {
    "least_squares": lambda y, X, L: least_squares(y, X),
    "logistica":     lambda y, X, L: logistic_regression(y, X, np.zeros(X.shape[1]),
                                                         MAX_ITERS, FATTORE_PASSO / L),
}
LOGISTICI = {"logistica"}

risultati = {nome: [] for nome in METODI}
for k in range(len(folds)):
    tr, va = split_fold(folds, k)
    t0 = time.perf_counter()
    p, X_tr = fit_transform_prep(x_base[tr], cols_base)
    X_va = transform_prep(x_base[va], p)
    X_tr, X_va = X_tr.astype(np.float64), X_va.astype(np.float64)
    y_tr, y_va = y01[tr], y01[va]
    L = costante_lipschitz(X_tr)
    print(f"fold {k+1}: preprocessing {time.perf_counter() - t0:.1f}s, L = {L:.2f}")

    for nome, allena in METODI.items():
        t0 = time.perf_counter()
        w, loss_tr = allena(y_tr, X_tr, L)
        tempo = time.perf_counter() - t0
        s_tr, s_va = X_tr @ w, X_va @ w
        if nome in LOGISTICI:
            s_tr, s_va = sigmoid(s_tr), sigmoid(s_va)
            loss_va = calculate_logistic_loss(y_va, X_va, w)
        else:
            loss_va = compute_loss(y_va, X_va, w)
        soglia, f1_tr = scegli_soglia(y_tr, s_tr)              # soglia scelta SUL TRAINING
        m = metriche(y_va, (s_va >= soglia).astype(int))       # metriche SUL VALIDATION
        risultati[nome].append(dict(tempo=tempo, loss_tr=loss_tr, loss_va=loss_va,
                                    soglia=soglia, f1_tr=f1_tr, **m))
        print(f"   {nome:14s} {tempo:6.1f}s   soglia {soglia:.3f}   F1 val {m['f1']:.4f}")

# --- riepilogo: media (± dev. std) sui fold ---
print(f"\n{'metodo':14s} {'tempo [s]':>9s} {'loss tr':>8s} {'loss val':>8s} {'soglia':>15s} "
      f"{'F1 tr':>6s} {'F1 val':>15s} {'prec':>6s} {'rec':>6s} {'acc':>6s}")
for nome, r in risultati.items():
    col = lambda c: np.array([x[c] for x in r])
    print(f"{nome:14s} {col('tempo').mean():9.1f} {col('loss_tr').mean():8.4f} {col('loss_va').mean():8.4f} "
          f"{col('soglia').mean():7.3f} ± {col('soglia').std():.3f} {col('f1_tr').mean():6.3f} "
          f"{col('f1').mean():8.4f} ± {col('f1').std():.4f} "
          f"{col('precision').mean():6.3f} {col('recall').mean():6.3f} {col('accuracy').mean():6.3f}")

# --- confronto appaiato: stessa divisione in fold per i due metodi ---
f1_ls  = np.array([x["f1"] for x in risultati["least_squares"]])
f1_log = np.array([x["f1"] for x in risultati["logistica"]])
delta = f1_log - f1_ls
print("\nF1 logistica - F1 least_squares, fold per fold:", np.round(delta, 4))
print(f"differenza media {delta.mean():+.4f}   dev. std delle differenze {delta.std():.4f}")
if abs(delta.mean()) > delta.std():
    print("--> la differenza SUPERA il rumore:", "vince la logistica" if delta.mean() > 0 else "vince least_squares")
else:
    print("--> la differenza è dentro il rumore: si sceglie il modello più semplice (least_squares)")

fold 1: preprocessing 9.9s, L = 8.29
   least_squares     0.3s   soglia 0.227   F1 val 0.4223
   logistica        57.3s   soglia 0.195   F1 val 0.4273
fold 2: preprocessing 9.8s, L = 8.28
   least_squares     0.5s   soglia 0.225   F1 val 0.4217
   logistica        56.3s   soglia 0.194   F1 val 0.4310
fold 3: preprocessing 9.6s, L = 8.26
   least_squares     0.3s   soglia 0.223   F1 val 0.4173
   logistica        54.2s   soglia 0.195   F1 val 0.4259
fold 4: preprocessing 9.6s, L = 8.26
   least_squares     0.2s   soglia 0.223   F1 val 0.4175
   logistica        55.2s   soglia 0.198   F1 val 0.4260
fold 5: preprocessing 9.8s, L = 8.25
   least_squares     0.4s   soglia 0.223   F1 val 0.4219
   logistica        56.9s   soglia 0.187   F1 val 0.4271

metodo         tempo [s]  loss tr loss val          soglia  F1 tr          F1 val   prec    rec    acc
least_squares        0.3   0.0330   0.0331   0.224 ± 0.002  0.422   0.4201 ± 0.0023  0.350  0.526  0.872
logistica           56.0   0.2209   

## Step 5: interazioni tra variabili

Il modello attuale è **additivo**: ogni variabile sposta il punteggio $x^\top w$ sempre della stessa quantità, qualunque siano le altre. Ma il rischio cardiaco si **moltiplica**: il fumo pesa molto di più a 70 anni che a 30. Per vederlo serve un'**interazione**, cioè una nuova colonna $x_a \cdot x_b$ con un suo peso $w_{ab}$:

$$s = \dots + w_a x_a + w_b x_b + w_{ab}\, x_a x_b \qquad\Rightarrow\qquad \text{effetto di } x_a = w_a + w_{ab}\, x_b$$

Il modello resta lineare nei pesi: cambiano solo le colonne, e i metodi di `implementations.py` si usano così come sono.

Proviamo **due insiemi di coppie**:

1. **guidate dai dati**: si allena la logistica attuale, si prendono le **10 colonne più importanti** e si costruiscono tutte le coppie tra loro;
2. **mediche**: età × sesso, età × diabete, età × fumo, età × salute generale, diabete × ipertensione, ipertensione × colesterolo alto (per una categorica si usano tutti i suoi livelli one-hot).

**Importanza di una colonna.** Solo le continue sono standardizzate; i one-hot valgono 0/1. Un livello raro può quindi avere un $|w|$ grande pur contando poco. Usiamo $|w_j| \cdot \text{dev.std}(x_j)$, cioè il peso che la colonna avrebbe se fosse standardizzata. Si tiene al massimo **una colonna per variabile** e si escludono bias, indicatori di NaN e livelli `=NaN`.

**Pulizia dei prodotti.** Si tolgono i prodotti **costanti** (due livelli dello stesso one-hot non sono mai 1 insieme: prodotto sempre 0) e quelli **identici a una colonna già presente**: renderebbero $X^\top X$ singolare e `least_squares` fallirebbe. I prodotti rimasti si standardizzano con media e dev. std **del training**.

**Niente fuga di informazione.** La scelta delle colonne usa i pesi della logistica, quindi va fatta **dentro ogni fold**, solo sul training (come `fit_transform_prep`).

In [ ]:
# =====================================================================
# Step 5: costruzione delle interazioni
# =====================================================================
K_VARIABILI = 10          # colonne più importanti da cui costruire le coppie "guidate dai dati"
COPPIE_MEDICHE = [("_AGE80", "SEX"), ("_AGE80", "DIABETE3"), ("_AGE80", "_SMOKER3"),
                  ("_AGE80", "GENHLTH"), ("DIABETE3", "BPHIGH4"), ("BPHIGH4", "TOLDHI2")]

def variabile_origine(nome):
    """'log AVEDRNK2' -> 'AVEDRNK2',  'GENHLTH=4' -> 'GENHLTH',  'NaN[_BMI5]' -> '_BMI5'."""
    if nome.startswith("NaN["):
        return nome[4:].split("]")[0].split(" ")[0]
    return nome.removeprefix("log ").split("=")[0]

def importanza(w, X):
    """|w_j| * dev. std(X_j): il peso che avrebbe la colonna j se fosse standardizzata."""
    return np.abs(w) * X.std(axis=0)

def scegli_colonne(w, X, nomi, tipo, k=K_VARIABILI):
    """Le k colonne più importanti, al massimo una per variabile di origine.
    Si escludono bias, indicatori di NaN e livelli '=NaN' (interazioni poco interpretabili)."""
    imp = importanza(w, X)
    scelte, viste = [], set()
    for j in np.argsort(-imp):
        if tipo[j] in ("bias", "ind") or nomi[j].endswith("=NaN"):
            continue
        v = variabile_origine(nomi[j])
        if v not in viste:
            scelte.append(int(j))
            viste.add(v)
        if len(scelte) == k:
            break
    return scelte

def colonne_di(variabile, nomi, tipo):
    """Colonne di una variabile: la continua, oppure tutti i livelli del one-hot (tranne '=NaN')."""
    return [j for j, (n, t) in enumerate(zip(nomi, tipo))
            if t in ("cont", "oh") and not n.endswith("=NaN") and variabile_origine(n) == variabile]

def coppie_da_colonne(colonne, nomi):
    """Tutte le coppie tra le colonne scelte."""
    return [(a, b) for i, a in enumerate(colonne) for b in colonne[i + 1:]
            if variabile_origine(nomi[a]) != variabile_origine(nomi[b])]

def coppie_da_variabili(coppie_var, nomi, tipo):
    """Per ogni coppia di variabili (A, B): ogni colonna di A per ogni colonna di B."""
    return [(a, b) for va_, vb in coppie_var
            for a in colonne_di(va_, nomi, tipo) for b in colonne_di(vb, nomi, tipo)]

def fit_interazioni(X, coppie):
    """Prodotti x_a * x_b, standardizzati con media e dev. std del TRAINING.
    Si tolgono i prodotti costanti e quelli identici a uno dei due fattori."""
    Z = np.column_stack([X[:, a] * X[:, b] for a, b in coppie])
    copia = np.array([np.array_equal(Z[:, i], X[:, a]) or np.array_equal(Z[:, i], X[:, b])
                      for i, (a, b) in enumerate(coppie)])
    tieni = (Z.std(axis=0) > 0) & ~copia
    Z = Z[:, tieni]
    pi = {"coppie": [c for c, t in zip(coppie, tieni) if t], "mu": Z.mean(axis=0), "sd": Z.std(axis=0)}
    return pi, (Z - pi["mu"]) / pi["sd"]

def transform_interazioni(X, pi):
    """Applica a validation / test le interazioni imparate sul training."""
    Z = np.column_stack([X[:, a] * X[:, b] for a, b in pi["coppie"]])
    return (Z - pi["mu"]) / pi["sd"]

def nomi_interazioni(pi, nomi):
    return [f"{nomi[a]} x {nomi[b]}" for a, b in pi["coppie"]]

In [ ]:
# =====================================================================
# Prova sul fold 1: quali colonne vengono scelte e quante interazioni si creano
# =====================================================================
tr, va = split_fold(folds, 0)
p, X_tr = fit_transform_prep(x_base[tr], cols_base)
X_va = transform_prep(x_base[va], p)
X_tr, X_va = X_tr.astype(np.float64), X_va.astype(np.float64)
y_tr, y_va = y01[tr], y01[va]

t0 = time.perf_counter()
w_base, _ = logistic_regression(y_tr, X_tr, np.zeros(X_tr.shape[1]), MAX_ITERS, FATTORE_PASSO / costante_lipschitz(X_tr))
print(f"logistica di partenza: {time.perf_counter() - t0:.1f}s\n")

colonne = scegli_colonne(w_base, X_tr, p["nomi"], p["tipo"])
imp = importanza(w_base, X_tr)
print(f"le {K_VARIABILI} colonne più importanti:")
for j in colonne:
    print(f"   {p['nomi'][j]:16s}  w {w_base[j]:+.3f}   importanza {imp[j]:.3f}")

for nome, coppie in [("guidate dai dati", coppie_da_colonne(colonne, p["nomi"])),
                     ("mediche", coppie_da_variabili(COPPIE_MEDICHE, p["nomi"], p["tipo"]))]:
    pi, _ = fit_interazioni(X_tr, coppie)
    print(f"\ninterazioni {nome}: {len(coppie)} coppie, {len(pi['coppie'])} tenute (tolte costanti / copie)")
    print("   " + ", ".join(nomi_interazioni(pi, p["nomi"])))

### Cross-validation: modello attuale vs modello con interazioni

Stessi 5 fold, stessa logistica ($\gamma = 16/L$, 1000 passi, con $L$ ricalcolato sulla matrice allargata) e stessa scelta della soglia sul training. Per ogni fold si allenano **6 modelli**: least_squares e logistica, ciascuno su

- **base**: le ~256 colonne attuali;
- **+ dati**: base + interazioni tra le 10 colonne più importanti (scelte con la logistica base **di quel fold**);
- **+ mediche**: base + le interazioni mediche.

Poi il **confronto appaiato**, fold per fold, di ogni variante con la sua base (least_squares con least_squares, logistica con logistica), con la stessa regola di prima: la differenza conta solo se supera il rumore. Tempo previsto: circa 13 minuti.

In [ ]:
# =====================================================================
# CV a 5 fold: base vs + interazioni (dati / mediche), least_squares e logistica
# =====================================================================
def allena_e_valuta(nome, X_tr, X_va, y_tr, y_va):
    """Allena least_squares o logistica, sceglie la soglia sul training, misura sul validation."""
    t0 = time.perf_counter()
    if nome == "logistica":
        w, loss_tr = logistic_regression(y_tr, X_tr, np.zeros(X_tr.shape[1]),
                                         MAX_ITERS, FATTORE_PASSO / costante_lipschitz(X_tr))
        s_tr, s_va = sigmoid(X_tr @ w), sigmoid(X_va @ w)
    else:
        w, loss_tr = least_squares(y_tr, X_tr)
        s_tr, s_va = X_tr @ w, X_va @ w
    tempo = time.perf_counter() - t0
    soglia, f1_tr = scegli_soglia(y_tr, s_tr)                    # soglia scelta SUL TRAINING
    m = metriche(y_va, (s_va >= soglia).astype(int))             # metriche SUL VALIDATION
    return w, dict(tempo=tempo, loss_tr=loss_tr, soglia=soglia, f1_tr=f1_tr, colonne=X_tr.shape[1], **m)

VARIANTI = ["base", "+ dati", "+ mediche"]
risultati_int = {(mod, var): [] for mod in ["least_squares", "logistica"] for var in VARIANTI}
scelte_per_fold = []

for k in range(len(folds)):
    tr, va = split_fold(folds, k)
    t0 = time.perf_counter()
    p, X_tr = fit_transform_prep(x_base[tr], cols_base)
    X_va = transform_prep(x_base[va], p)
    X_tr, X_va = X_tr.astype(np.float64), X_va.astype(np.float64)
    y_tr, y_va = y01[tr], y01[va]
    print(f"fold {k+1}: preprocessing {time.perf_counter() - t0:.1f}s")

    # modelli base: la logistica base serve anche per scegliere le colonne
    for mod in ["least_squares", "logistica"]:
        w, r = allena_e_valuta(mod, X_tr, X_va, y_tr, y_va)
        risultati_int[(mod, "base")].append(r)
        if mod == "logistica":
            w_base = w
    colonne = scegli_colonne(w_base, X_tr, p["nomi"], p["tipo"])
    scelte_per_fold.append([p["nomi"][j] for j in colonne])

    coppie = {"+ dati":    coppie_da_colonne(colonne, p["nomi"]),
              "+ mediche": coppie_da_variabili(COPPIE_MEDICHE, p["nomi"], p["tipo"])}
    for var, cp in coppie.items():
        pi, Z_tr = fit_interazioni(X_tr, cp)
        XI_tr, XI_va = np.hstack([X_tr, Z_tr]), np.hstack([X_va, transform_interazioni(X_va, pi)])
        for mod in ["least_squares", "logistica"]:
            _, r = allena_e_valuta(mod, XI_tr, XI_va, y_tr, y_va)
            risultati_int[(mod, var)].append(r)
    for (mod, var), r in risultati_int.items():
        print(f"   {mod:14s} {var:10s} {r[-1]['colonne']:4d} col  {r[-1]['tempo']:6.1f}s   F1 val {r[-1]['f1']:.4f}")

# --- colonne scelte: sono stabili tra i fold? ---
print("\ncolonne scelte (quanti fold su 5):")
for nome, n in sorted({c: sum(c in s for s in scelte_per_fold) for s in scelte_per_fold for c in s}.items(),
                      key=lambda t: -t[1]):
    print(f"   {nome:18s} {n}")

# --- riepilogo: media (± dev. std) sui fold ---
print(f"\n{'metodo':14s} {'variante':10s} {'col':>4s} {'tempo [s]':>9s} {'F1 tr':>6s} {'F1 val':>15s} "
      f"{'prec':>6s} {'rec':>6s} {'acc':>6s}")
for (mod, var), r in risultati_int.items():
    col = lambda c: np.array([x[c] for x in r])
    print(f"{mod:14s} {var:10s} {col('colonne').mean():4.0f} {col('tempo').mean():9.1f} {col('f1_tr').mean():6.3f} "
          f"{col('f1').mean():8.4f} ± {col('f1').std():.4f} "
          f"{col('precision').mean():6.3f} {col('recall').mean():6.3f} {col('accuracy').mean():6.3f}")

# --- confronto appaiato con la base dello stesso metodo ---
print()
for mod in ["least_squares", "logistica"]:
    f1_base = np.array([x["f1"] for x in risultati_int[(mod, "base")]])
    for var in ["+ dati", "+ mediche"]:
        delta = np.array([x["f1"] for x in risultati_int[(mod, var)]]) - f1_base
        esito = ("SUPERA il rumore: " + ("le interazioni aiutano" if delta.mean() > 0 else "le interazioni peggiorano")
                 if abs(delta.mean()) > delta.std() else "dentro il rumore: si tiene il modello base")
        print(f"{mod:14s} {var:10s} delta F1 per fold {np.round(delta, 4)}   "
              f"media {delta.mean():+.4f}  dev. std {delta.std():.4f}  --> {esito}")